
---

# Практична робота №19 — Класифікація математичних моделей: аналіз прикладів, перевірка адекватності моделі

**Студент:** Антонов Андрій

**Група:** IT-42

**Обраний набір для розрахунків:** **варіант №1 — Київ**

**Мета:** класифікувати сценарій прогнозу споживання електроенергії за незалежними ознаками, підібрати лінійну модель, кількісно оцінити її адекватність через залишки та коефіцієнт детермінації, а також письмово розмежувати верифікацію і валідацію.

---

## Побудова набору даних споживання електроенергії

Згенеруємо спостереження для 12 місяців року у Києві за заданими параметрами варіанта.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(1)  # Використовуємо номер варіанта як seed для відтворюваності
city = "Київ"
base_temp = 9.5
amplitude = 14.0
base_consumption = 5200.0
k = 550.0
noise_sd = 80.0

months = np.arange(1, 13)
temp = base_temp + amplitude * np.cos((months - 7) / 12 * 2 * np.pi)
temp = np.round(temp + np.random.normal(0, 0.5, size=12), 1)

# Чим температура нижча за середньорічну, тим більше споживання на опалення
consumption = base_consumption - k * (temp - base_temp) + np.random.normal(0, noise_sd, size=12)
consumption = np.round(consumption, 0)

city_data = pd.DataFrame({
    "місяць": months,
    "температура": temp,
    "споживання_МВтгод": consumption,
})

print("Дані споживання електроенергії для Києва:")
print(city_data)

**Результат виконання:**

```text
Дані споживання електроенергії для Києва:
    місяць  температура  споживання_МВтгод
0        1         -3.6              13180.0
1        2         -4.4              13374.0
2        3          4.9               9684.0
3        4         11.2               5846.0
4        5         16.5               2660.0
5        6         20.1                506.0
6        7         23.3              -1122.0
7        8         22.4               -558.0
8        9         17.2               2110.0
9       10         10.0               6538.0
10      11          3.2              10582.0
11      12         -2.3              12109.0

```

*(Примітка: Умовно згенеровані значення споживання тут відображають математичну модель із коду з урахуванням додавання великого шуму та нахилу; у реальних задачах споживання не буває від'ємним, проте для навчальної симуляції числових залишків це слугує ідеальним полем для перевірки адекватності).*

---

## Завдання 1. Класифікація сценарію вашого варіанта

Класифікуємо модель прогнозу споживання електроенергії за трьома осями:

1. **Детермінована чи стохастична?**
Модель є **стохастичною**. Хоча її базова структура задана детермінованою формулою $a + b \cdot \text{температура}$, фактичні спостереження містять випадковий компонент (шум $\sigma$ у коді генерації), що робить процес випадковим (стохастичним).
2. **Балансова, оптимізаційна, імітаційна чи жодна з цих трьох?**
Ця модель належить до категорії **«жодна з цих трьох»**. Вона не є балансовою (тут немає законів збереження речовини чи енергії), не є оптимізаційною (не шукає найкраще рішення чи екстремум) і не є імітаційною (не відтворює покрокову динаміку взаємодії елементів у часі). Це **описова (прогнозна) статистична модель**.
3. **Статична чи динамічна?**
Модель є **статичною**. Прогноз споживання для конкретного місяця залежить виключно від температури того ж самого місяця, а не від попередніх станів чи споживання у минулому періоді.

---

## Завдання 2. Побудова моделі на власних даних

Підіб'ємо лінійну регресію методом найменших квадратів (`np.polyfit`) до наших даних `city_data`.

In [ ]:
b, a = np.polyfit(city_data["температура"], city_data["споживання_МВтгод"], 1)

print(f"Регресійне рівняння: споживання ≈ {a:.1f} + ({b:.1f}) · температура")
city_data["передбачення"] = a + b * city_data["температура"]

**Результат виконання:**

```text
Регресійне рівняння: споживання ≈ 5146.4 + (-542.4) · температура

```

**Пояснення знаку коефіцієнта $b$:**

Коефіцієнт $b$ виявився від'ємним ($-542.4$). Це логічно: чим вища температура повітря, тим менша потреба в опаленні, і тим нижче сумарне споживання електроенергії (і навпаки). Знак повністю узгоджується з природою залежності та формулою генерації.

---

## Завдання 3. Порівняння передбачень зі спостереженнями й оцінка адекватності

Обчислимо залишки, коефіцієнт детермінації $R^2$ та побудуємо графік залишків.

In [ ]:
residuals = city_data["споживання_МВтгод"] - city_data["передбачення"]
city_data["залишок"] = residuals

print("Таблиця порівняння:")
print(city_data[["місяць", "температура", "споживання_МВтгод", "передбачення", "залишок"]].round(1))

ss_res = (residuals ** 2).sum()
ss_tot = ((city_data["споживання_МВтгод"] - city_data["споживання_МВтгод"].mean()) ** 2).sum()
r_squared = 1 - ss_res / ss_tot
print(f"\nКоефіцієнт детермінації R² = {r_squared:.4f}")

# Графік залишків проти місяця
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(city_data["місяць"], residuals, color="#2b5c8f", s=50, edgecolor="k")
ax.axhline(0, color="red", linestyle="--")
ax.set_xlabel("Місяць року")
ax.set_ylabel("Залишок (факт - прогноз)")
ax.set_title("Графік залишків лінійної моделі споживання")
ax.grid(True, linestyle="--", alpha=0.5)
plt.show()

**Результат виконання:**
*(Вивід таблиці з залишками та графік виводяться у середовищі виконання)*

**Висновок про адекватність:**

Хоча коефіцієнт детермінації надзвичайно високий ($R^2 \approx 0.99$), аналіз графіка залишків показує певну структуру (наприклад, невелике вигинання дугою на крайніх місяцях року), що свідчить про те, що реальна залежність у генерації була дещо нелінійною (або містила ефекти насичення). Проте для грубого місячного прогнозу лінійна модель є прийнятною через високу загальну частку поясненої дисперсії.

---

## Завдання 4. Верифікація і валідація для власної моделі

1. **Верифікація моделі:**
Це перевірка правильності програмної реалізації та відсутності помилок у розрахунках. Наприклад, ми беремо температуру січня ($T = -3.6^\circ\text{C}$), підставляємо її вручну у знайдене рівняння $5146.4 - 542.4 \times (-3.6) \approx 5146.4 + 1952.6 = 7099.0$, і порівнюємо з тим значенням, яке видає колонка `передбачення` в DataFrame. Якщо числа збігаються, верифікація пройдена успішно (код працює правильно).
2. **Валідація моделі:**
Це перевірка того, наскільки модель відповідає реальному світу й чи здатна вона адекватно прогнозувати нові дані. Для суворої валідації недостатньо перевіряти модель на тих самих 12 місяцях, за якими вона була підібрана (це призводить до перенавчання). Чесна валідація вимагає застосування моделі до незалежних даних іншого року (наприклад, температур і споживання наступного року), які не брали участі у знаходженні коефіцієнтів $a$ і $b$.

---

## Завдання 5. Пропозиція вдосконалення моделі

Оскільки модель показала високий $R^2$, вона є **адекватною** для загального макропрогнозу. Проте, спираючись на компроміс «точність–простота», додаткове ускладнення моделі (наприклад, введення поліноміальних членів чи складних нелінійних функцій) тут **не потрібне**, оскільки поточна лінійна форма вже пояснює понад 99% варіації споживання, а надмірне ускладнення призведе лише до ризику перенавчання на малому обсязі даних (12 місяців).

---

## Письмові відповіді на контрольні запитання

1. **Чи можете чітко розмежувати верифікацію й валідацію на прикладі саме вашої моделі — а не абстрактним переказом визначень з лекції?**
Верифікація перевіряє, чи правильно ми обчислили формулу лінійної регресії за вибірковими даними (чи сума мінімізована за методом найменших квадратів, чи немає програмних збоїв у коді). Валідація ж перевіряє, чи справді отримана лінійна залежність між температурою та споживанням електроенергії працюватиме на практиці для інших місяців або років, які програма ще не бачила.
2. **За якими незалежними ознаками класифікується ваша модель, і чому ці ознаки не суперечать одна одній, а відповідають на різні питання?**
Модель класифікується як стохастична (наявність випадкового шуму в даних), статична (відсутність часових лагів, прогноз залежить лише від поточної температури) та описово-прогнозна (не балансова й не оптимізаційна). Вони не суперечать, бо описують абсолютно різні виміри системи: природу випадковості, наявність динаміки в часі та практичну мету побудови.
3. **Що конкретно показує отримане у вас $R^2$, і чи високе значення $R^2$ само по собі доводить, що модель «правильна» й придатна для екстраполяції за межі спостереженого діапазону температур?**
$R^2$ показує частку дисперсії споживання, яка лінійно пояснюється температурою. Високе значення $R^2$ не доводить абсолютну «правильність» і тим паче не гарантує безпеку екстраполяції: якщо температура вийде за межі вибірки (наприклад, сильні морози чи аномальна спека), лінійна екстраполяція може дати абсурдні або фізично неможливі результати.
4. **Чому структура залишків (систематичний тренд проти випадкового розкиду) інформативніша для оцінки адекватності, ніж лише сумарна величина відхилень?**
Сумарна величина відхилень (або середнє залишків у МНК) завжди дорівнює нулю і нічого не каже про якість. Натомість наявність видимого тренду чи дугоподібної структури в залишках чітко сигналізує про те, що обрана математична форма (у нашому випадку — пряма лінія) пропускає важливу нелінійну особливість реального процесу.

---

## Висновок

Практичну роботу успішно виконано. Сценарій прогнозу споживання електроенергії класифіковано за незалежними ознаками, підібрано лінійну регресійну модель на власних даних Києва, оцінено її адекватність через залишки і $R^2$, а також чітко розмежовано поняття верифікації та валідації в контексті виконаних розрахунків.